<a href="https://colab.research.google.com/github/amhaiskar0921/BTTAIAmazonProject/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# PIP INSTALL NEEDED FOR QUANTIZING LLM MODEL WEIGHTS IN ADVANCED APPROACH
!pip install -U bitsandbytes # Only run ONCE and restart runtime after it finishes running

# DATASET

In [ ]:
from huggingface_hub import hf_hub_download
import json

import pandas as pd
from datasets import Dataset, DatasetDict

# Example code for loading dataset. We manually recreated a dataframe with 5 random samples from it instead for faster loading in this demo.

'''
# Loading dataset
file_path = hf_hub_download(
    repo_id="OSS-forge/HumanVsAICode",
    filename="python_dataset.jsonl",
    repo_type="dataset"
)

print("File downloaded to:", file_path)

N = 5

data = []
with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):

        data.append(json.loads(line))

# Creating Dataframe and organizing data
df = pd.DataFrame(data)

code_cols = ['human_code', 'chatgpt_code', 'dsc_code', 'qwen_code']
df_long = df.melt(id_vars=['docstring'], value_vars=code_cols,
                  var_name='writer_type', value_name='code')

df_long['writer'] = df_long['writer_type'].map({
    'human_code': 'human',
    'chatgpt_code': 'gpt',
    'dsc_code': 'dsc',
    'qwen_code': 'qwen'
})
df_long['is_ai'] = (df_long['writer'] != 'human').astype(int)

demo_samples = df_long.sample(n=5, random_state=42)
demo_samples[["writer", "is_ai", "code"]]

codes = demo_samples["code"].tolist()
labels = demo_samples["is_ai"].tolist()
writers = demo_samples["writer"].tolist()
'''


'\n# Loading dataset\nfile_path = hf_hub_download(\n    repo_id="OSS-forge/HumanVsAICode",\n    filename="python_dataset.jsonl",\n    repo_type="dataset"\n)\n\nprint("File downloaded to:", file_path)\n\nN = 5\n\ndata = []\nwith open(file_path, "r", encoding="utf-8") as f:\n    for i, line in enumerate(f):\n\n        data.append(json.loads(line))\n\n# Creating Dataframe and organizing data\ndf = pd.DataFrame(data)\n\ncode_cols = [\'human_code\', \'chatgpt_code\', \'dsc_code\', \'qwen_code\']\ndf_long = df.melt(id_vars=[\'docstring\'], value_vars=code_cols,\n                  var_name=\'writer_type\', value_name=\'code\')\n\ndf_long[\'writer\'] = df_long[\'writer_type\'].map({\n    \'human_code\': \'human\',\n    \'chatgpt_code\': \'gpt\',\n    \'dsc_code\': \'dsc\',\n    \'qwen_code\': \'qwen\'\n})\ndf_long[\'is_ai\'] = (df_long[\'writer\'] != \'human\').astype(int)\n\ndemo_samples = df_long.sample(n=5, random_state=42)\ndemo_samples[["writer", "is_ai", "code"]]\n\ncodes = demo_samples[

The below cell is a recreation of the 5 randomly selected samples that would be generated if the above cell was uncommented.

In [ ]:
data = [
    {
        "writer_type": "human_code",
        "code": "def append_dict_values(list_of_dicts, keys=None):\n    if keys is None:\n        keys = list(list_of_dicts[0].keys())\n    dict_of_lists = DefaultOrderedDict(list)\n    for d in list_of_dicts:\n        for k in keys:\n            dict_of_lists[k].append(d[k])\n    return dict_of_lists",
        "writer": "human",
        "is_ai": 0
    },
    {
        "writer_type": "qwen_code",
        "code": "def _sensoryComputeLearningMode(self, anchorInput):\n    learning_mode = np.mean(anchorInput)\n    return learning_mode",
        "writer": "qwen",
        "is_ai": 1
    },
    {
        "writer_type": "qwen_code",
        "code": "def _set_logical_chassis(self, v, load=False):\n    if hasattr(v, '_utype'):\n        v = v._utype(v)\n    try:\n        t = YANGDynClass(v, base=logical_chassis.logical_chassis, is_container='container', presence=True, yang_name='logical_chassis', rest_name='logical_chassis', parent=self, path_helper=self._path_helper, extmethods=self._extmethods, register_paths=True, extensions={u'tailf-common': {u'cli-full-command': None, u'cli-drop-node-name': None}}, namespace='urn:brocade.com:mgmt:brocade-rbridge', defining_module='brocade-rbridge', yang_type='container', is_config=True)\n        self.__logical_chassis = t\n        if load is False:\n            self._set()\n    except (TypeError, ValueError):\n        raise ValueError({'error-string': '%Error: /rbridge_id/logical_chassis', 'error-code': 'invalid-value'})",
        "writer": "qwen",
        "is_ai": 1
    },
    {
        "writer_type": "human_code",
        "code": "def export(self):\n        return {'id': self.id, 'name': self.name, 'artist': self._artist_name,\n                'artist_id': self._artist_id, 'album': self._album_name,\n                'album_id': self._album_id, 'track': self.track,\n                'duration': self.duration, 'popularity': self.popularity,\n                'cover': self._cover_url}",
        "writer": "human",
        "is_ai": 0
    },
    {
        "writer_type": "human_code",
        "code": "def add_relations(self, relations):\n        for source, destination in relations:\n            self.add_relation(source, destination)",
        "writer": "human",
        "is_ai": 0
    }
]

sample_df = pd.DataFrame(data)

sample_df

,writer_type,code,writer,is_ai
0,human_code,"def append_dict_values(list_of_dicts, keys=Non...",human,0
1,qwen_code,"def _sensoryComputeLearningMode(self, anchorIn...",qwen,1
2,qwen_code,"def _set_logical_chassis(self, v, load=False):...",qwen,1
3,human_code,def export(self):\n return {'id': self....,human,0
4,human_code,"def add_relations(self, relations):\n f...",human,0


# BASELINE METHOD

Created a feature extractor that uses abstract syntax trees to find structural characteristics of Python code. ASTs represent the syntactic structure of the program as a hierarchical tree allowing you to analyze independent of formatting and whitespace. From this representation we extracted features in 5 main categories: control flow, error_handling, structure, identifier, comment.

In [ ]:
import ast
import numpy as np
import tokenize
import io

class ASTFeatureExtractor(ast.NodeVisitor):

  def __init__(self):

    self.FEATURE_NAMES = [

      'if',
      'for',
      'while',
      'return',
      'break',
      'continue',

      'error_handling',

      'functions',
      'classes',
      'docstrings',

      'unique_identifiers',
      'avg_identifier_length',
      'single_char_identifiers',

      'comment_count',
      'avg_comment_length',
      'comment_density',
      'todo_comments',

      'max_depth'

    ]

    self.features = {

      'if': 0,
      'for': 0,
      'while': 0,
      'return': 0,
      'break': 0,
      'continue': 0,
      'error_handling': 0,
      'functions': 0,
      'classes': 0,
      'docstrings': 0
    }

    self.identifiers = []

    self.curr_depth = 0
    self.max_depth = 0

    self.comment_count = 0
    self.comment_lengths = []
    self.todo_comments = 0

    self.total_lines = 0


  def generic_visit(self, node):
    self.curr_depth += 1
    self.max_depth = max(self.max_depth, self.curr_depth)
    super().generic_visit(node)
    self.curr_depth -= 1


  #control flow
  def visit_If(self, node):
    self.features["if"] += 1
    super().generic_visit(node)

  def visit_For(self, node):
    self.features["for"] += 1
    super().generic_visit(node)

  def visit_While(self, node):
    self.features["while"] += 1
    super().generic_visit(node)

  def visit_Return(self, node):
    self.features["return"] += 1
    super().generic_visit(node)

  def visit_Break(self, node):
    self.features["break"] += 1

  def visit_Continue(self, node):
    self.features["continue"] += 1

  #error handling
  def visit_Try(self, node):
    self.features["error_handling"] += 1
    super().generic_visit(node)

  def visit_Assert(self, node):
    self.features["error_handling"] += 1
    super().generic_visit(node)

  def visit_Raise(self, node):
    self.features["error_handling"] += 1
    super().generic_visit(node)

  #structure
  def visit_FunctionDef(self, node):
        self.features["functions"] += 1

        if ast.get_docstring(node):
            self.features["docstrings"] += 1

        super().generic_visit(node)

  def visit_ClassDef(self, node):
      self.features["classes"] += 1

      if ast.get_docstring(node):
          self.features["docstrings"] += 1

      super().generic_visit(node)

  #comments
  def extract_comments(self, source_code):
      self.comment_count = 0
      self.comment_lengths = []
      self.todo_comments = 0

      tokens = tokenize.generate_tokens(io.StringIO(source_code).readline)

      for tok_type, tok_string, _, _, _ in tokens:
          if tok_type == tokenize.COMMENT:
              self.comment_count += 1

              cleaned = tok_string.lstrip("#").strip()
              self.comment_lengths.append(len(cleaned))

              if "todo" in cleaned.lower() or "fixme" in cleaned.lower():
                  self.todo_comments += 1

  #identifiers
  def visit_Name(self, node):
    self.identifiers.append(node.id)
    super().generic_visit(node)

  #feature vector
  def get_feature_vector(self):

    if self.identifiers:
      unique_identifiers = len(set(self.identifiers))
      identifier_lengths = [len(identifier) for identifier in set(self.identifiers)]
      avg_identifier_length = round(sum(identifier_lengths) / len(identifier_lengths) if len(identifier_lengths) > 0 else 0, 2)
      single_char_identifiers = sum(1 for name in self.identifiers if len(name) == 1)
    else:
      unique_identifiers = 0
      avg_identifier_length = 0
      single_char_identifiers = 0

    if self.comment_lengths:
            avg_comment_length = round(
                sum(self.comment_lengths) / len(self.comment_lengths),
                2
            )
    else:
            avg_comment_length = 0

    comment_density = round(
          self.comment_count / self.total_lines,
          4
      ) if self.total_lines > 0 else 0

    self.features["unique_identifiers"] = unique_identifiers
    self.features["avg_identifier_length"] = avg_identifier_length
    self.features["single_char_identifiers"] = single_char_identifiers
    self.features["comment_count"] = self.comment_count
    self.features["avg_comment_length"] = avg_comment_length
    self.features["comment_density"] = comment_density
    self.features["todo_comments"] = self.todo_comments
    self.features["max_depth"] = self.max_depth

    return [self.features.get(f, 0) for f in self.FEATURE_NAMES]

  def get_zero_vector(self):
        return [0.0] * len(self.FEATURE_NAMES)


  def extract(self, source_code):
        self.total_lines = len(source_code.splitlines())

        try:
            tree = ast.parse(source_code)
            self.visit(tree)
        except SyntaxError:
            return self.get_zero_vector()

        self.extract_comments(source_code)

        return self.get_feature_vector()


Combined two different TF-IDF vectorizers to obtain two complementary sets of lexical features: a token-level TF-IDF representation and a character-level 4-gram TF-IDF representation. The token-based features captures identifiers and keywords while the character-4-gram featyres captures stylistic patterns within identifiers or patterns across tokens.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack


class TFIDFExtractor:

    def __init__(self, max_token_features=5000, max_char_features=5000):

        self.token_vectorizer = TfidfVectorizer(
            token_pattern=r'\b\w+\b',
            max_features=max_token_features
        )

        self.char_vectorizer = TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(4, 4),
            max_features=max_char_features
        )

    def fit(self, code_list):

        self.token_vectorizer.fit(code_list)
        self.char_vectorizer.fit(code_list)
        return self

    def transform(self, code_list):

        token_features = self.token_vectorizer.transform(code_list)
        char_features = self.char_vectorizer.transform(code_list)

        return hstack([token_features, char_features])

    def get_feature_names(self):

        token_names = self.token_vectorizer.get_feature_names_out()
        char_names = self.char_vectorizer.get_feature_names_out()

        token_names = [f"tok_{t}" for t in token_names]
        char_names = [f"char_{c}" for c in char_names]

        return np.concatenate([token_names, char_names])

We then fit the tfidf_extractor on the dataset to find the top features. These vectorizers find thousands of tokens many of which only appear a few times in the entire dataset of code and contribute very little useful information. To reduce the dimensionality of the feature space and limit memory usage, we cut down to the top 400 (200 token and 200 character-4-gram).

We then use both feature extraction methods (TF-IDF and AST) and extract the their respective features before training the model on it.

The cell below downloads the pre-fitted TF-IDF extractor, feature scaler, and the pre-trained LogisticRegression Classifier from a HuggingFace repository.

In [ ]:
from huggingface_hub import hf_hub_download
import joblib

model_path = hf_hub_download(
    repo_id="kjahncke/simple-ai-code-detector",
    filename="ai_code_detector.joblib"
)

model_package = joblib.load(model_path)

clf = model_package["model"]
scaler = model_package["scaler"]
tfidf_extractor = model_package["tfidf_extractor"]

We process the test data using the same preprocessing pipeline descibed above for the training data. For each code sample, lexical features are extracted using TF-IDF and structural featyures are extracted using ASTs. The resulting features are then scaled using the previously fitted scaler.

In [ ]:
from scipy.sparse import hstack, csr_matrix

sample_df_tfidf = tfidf_extractor.transform(sample_df["code"])

sample_df_ast = np.array([
    ASTFeatureExtractor().extract(code)
    for code in sample_df["code"]
], dtype=np.float32)

sample_df_features = hstack([
    sample_df_tfidf,
    csr_matrix(sample_df_ast)
])

sample_df_scaled_features = scaler.transform(sample_df_features)

Now we can see how the model performs on each sample. in this small set of 5 randomly selected samples, the model achives an accuracy of 40%, much lower than its typical overall accuracy of 85%.

This discrepancy is likely due to the small sample size and distribution of samples in the dataset. The randomly selected samples happen to only include human and qwen code and from our broader testing Qwen code is misclassified as human-written 30.2% of the time while code generated by other models, such as DeepSeek are misclassified much less frequently (5.44%).

In [ ]:
for i, row in sample_df.iterrows():

    X_row = sample_df_scaled_features[i:i+1]

    pred = clf.predict(X_row)[0]
    probability = clf.predict_proba(X_row)[0][1]

    predicted_label = "AI" if pred else "Human"
    actual_label = row["writer"].title()

    print("=" * 60)
    print("Prediction:", predicted_label)
    print("Actual:", actual_label)
    print("AI probability:", probability)

Prediction: Human
Actual: Human
AI probability: 0.24992949503604742
Prediction: Human
Actual: Qwen
AI probability: 0.3603579154767565
Prediction: Human
Actual: Qwen
AI probability: 0.0304936168407453
Prediction: Human
Actual: Human
AI probability: 0.08505472641584329
Prediction: AI
Actual: Human
AI probability: 0.5772035105838763


# INTERMEDIATE METHOD

For the intermediate method, the AST features were changed slightly to differ from the baseline method.

In [ ]:
import ast
import tokenize
import io
import math
from collections import Counter
import numpy as np

class ASTFeatureExtractor(ast.NodeVisitor):

    CONTROL_NODES = (
        ast.If, ast.For, ast.While,
        ast.Try, ast.FunctionDef, ast.ClassDef
    )

    def __init__(self):

        self.FEATURE_NAMES = [
            # structural counts (normalized later)
            "if_count",
            "loop_count",
            "return_count",

            # structure
            "functions",
            "classes",
            "docstrings",

            # depth
            "max_control_depth",

            # function statistics
            "avg_function_length",
            "max_function_length",
            "functions_per_100_lines",

            # identifiers
            "unique_identifiers",
            "avg_identifier_length",
            "single_char_identifiers",
            "identifier_entropy",

            # error / defensive coding
            "error_handling_count",
            "error_handling_per_function",
            "empty_except_blocks",

            # comments
            "comment_count",
            "comment_density",
            "avg_comment_length",
            "todo_comments",

            # misc
            "pass_statements"
        ]

        self.reset()

    # ---------- state ----------

    def reset(self):
        self.features = Counter()
        self.identifiers = []

        self.function_lengths = []

        self.curr_depth = 0
        self.max_depth = 0

        self.comment_count = 0
        self.comment_lengths = []
        self.todo_comments = 0

        self.empty_except_blocks = 0
        self.pass_statements = 0

        self.total_lines = 0

    # ---------- depth handling ----------

    def generic_visit(self, node):
        is_control = isinstance(node, self.CONTROL_NODES)
        if is_control:
            self.curr_depth += 1
            self.max_depth = max(self.max_depth, self.curr_depth)

        super().generic_visit(node)

        if is_control:
            self.curr_depth -= 1

    # ---------- control flow ----------

    def visit_If(self, node):
        self.features["if_count"] += 1
        self.generic_visit(node)

    def visit_For(self, node):
        self.features["loop_count"] += 1
        self.generic_visit(node)

    def visit_While(self, node):
        self.features["loop_count"] += 1
        self.generic_visit(node)

    def visit_Return(self, node):
        self.features["return_count"] += 1
        self.generic_visit(node)

    # ---------- error handling ----------

    def visit_Try(self, node):
        self.features["error_handling_count"] += 1

        if not node.handlers or all(len(h.body) == 0 for h in node.handlers):
            self.empty_except_blocks += 1

        self.generic_visit(node)

    def visit_Assert(self, node):
        self.features["error_handling_count"] += 1
        self.generic_visit(node)

    def visit_Raise(self, node):
        self.features["error_handling_count"] += 1
        self.generic_visit(node)

    # ---------- structure ----------

    def visit_FunctionDef(self, node):
        self.features["functions"] += 1

        if ast.get_docstring(node):
            self.features["docstrings"] += 1

        if node.end_lineno:
            self.function_lengths.append(
                node.end_lineno - node.lineno + 1
            )

        self.generic_visit(node)

    def visit_ClassDef(self, node):
        self.features["classes"] += 1

        if ast.get_docstring(node):
            self.features["docstrings"] += 1

        self.generic_visit(node)

    def visit_Pass(self, node):
        self.pass_statements += 1

    # ---------- identifiers ----------

    def visit_Name(self, node):
        self.identifiers.append(node.id)
        self.generic_visit(node)

    # ---------- comments ----------

    def extract_comments(self, source_code):
        tokens = tokenize.generate_tokens(io.StringIO(source_code).readline)

        for tok_type, tok_string, _, _, _ in tokens:
            if tok_type == tokenize.COMMENT:
                self.comment_count += 1
                cleaned = tok_string.lstrip("#").strip()
                self.comment_lengths.append(len(cleaned))

                if "todo" in cleaned.lower() or "fixme" in cleaned.lower():
                    self.todo_comments += 1

    # ---------- feature assembly ----------

    def identifier_entropy(self):
        counts = Counter(self.identifiers)
        total = sum(counts.values())

        if total == 0:
            return 0.0

        return -sum(
            (c / total) * math.log2(c / total)
            for c in counts.values()
        )

    def get_feature_vector(self):

        avg_fn_len = np.mean(self.function_lengths) if self.function_lengths else 0
        max_fn_len = max(self.function_lengths) if self.function_lengths else 0

        unique_ids = len(set(self.identifiers))
        avg_id_len = (
            np.mean([len(i) for i in set(self.identifiers)])
            if self.identifiers else 0
        )
        single_char_ids = sum(
            1 for i in set(self.identifiers) if len(i) == 1
        )

        avg_comment_len = (
            np.mean(self.comment_lengths)
            if self.comment_lengths else 0
        )

        return [
            self.features["if_count"],
            self.features["loop_count"],
            self.features["return_count"],

            self.features["functions"],
            self.features["classes"],
            self.features["docstrings"],

            self.max_depth,

            round(avg_fn_len, 2),
            max_fn_len,
            round(self.features["functions"] / max(self.total_lines, 1) * 100, 4),

            unique_ids,
            round(avg_id_len, 2),
            single_char_ids,
            round(self.identifier_entropy(), 4),

            self.features["error_handling_count"],
            round(
                self.features["error_handling_count"] /
                max(self.features["functions"], 1), 4
            ),
            self.empty_except_blocks,

            self.comment_count,
            round(self.comment_count / max(self.total_lines, 1), 4),
            round(avg_comment_len, 2),
            self.todo_comments,

            self.pass_statements
        ]


    def extract(self, source_code):
        self.reset()
        self.total_lines = len(source_code.splitlines())

        try:
            tree = ast.parse(source_code)
            self.visit(tree)
        except SyntaxError:
            return [0.0] * len(self.FEATURE_NAMES)

        self.extract_comments(source_code)
        return self.get_feature_vector()

We combined the feature extractions with the CodeBert embeddings in the intermediate approach towards the end

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
model = AutoModel.from_pretrained("microsoft/codebert-base")

model.to(DEVICE)
model.eval()

for param in model.parameters():
    param.requires_grad = False


def codebert_features(code_list, batch_size=8, max_length=256):
    all_embeddings = []

    for i in range(0, len(code_list), batch_size):
        batch = code_list[i:i+batch_size]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(DEVICE)

        with torch.no_grad():
            outputs = model(**enc)

        # Mean pooling (masked)
        last_hidden = outputs.last_hidden_state  # [B, T, H]
        mask = enc.attention_mask.unsqueeze(-1)

        pooled = (last_hidden * mask).sum(1) / mask.sum(1)
        all_embeddings.append(pooled.cpu().numpy())

    return np.vstack(all_embeddings)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Here loads the intermediate approach models from hugging face which also contains the pca and scalers for normalization.

In [ ]:
from huggingface_hub import hf_hub_download
import numpy as np
import joblib

repo_id = "Peihan-Cui/ai-code-detector"

clf_path = hf_hub_download(repo_id=repo_id, filename="classifier.joblib")
pca_path = hf_hub_download(repo_id=repo_id, filename="pca.joblib")
scaler_path = hf_hub_download(repo_id=repo_id, filename="ast_scaler.joblib")

clf = joblib.load(clf_path)
pca = joblib.load(pca_path)
ast_scaler = joblib.load(scaler_path)

print("Model loaded!")

Model loaded!


Here we extract the AST features and the CodeBERT embeddings, scale and transform both, then pass it to the trained model for prediction. It outputs the predicted label and the probability between 0 to 1 where 0 is human and 1 is AI.

In [ ]:
extractor = ASTFeatureExtractor()

for i, row in sample_df.iterrows():

    code = row["code"]
    label = row["is_ai"]

    # AST features
    ast_features = extractor.extract(code)
    ast_features = ast_scaler.transform([ast_features])

    # CodeBERT embeddings
    cb = codebert_features([code])
    cb = pca.transform(cb)

    # combine
    import numpy as np
    X = np.hstack([ast_features, cb])

    # predict
    pred = clf.predict(X)[0]
    prob = clf.predict_proba(X)[0][1]

    print("=" * 60)
    print("True label :", "AI" if label else "Human")
    print("Prediction :", "AI" if pred else "Human")
    print("AI Likelihood:", round(prob, 3))

True label : Human
Prediction : AI
AI Likelihood: 0.918
True label : AI
Prediction : AI
AI Likelihood: 0.897
True label : AI
Prediction : AI
AI Likelihood: 0.675
True label : Human
Prediction : Human
AI Likelihood: 0.387
True label : Human
Prediction : Human
AI Likelihood: 0.443


# ADVANCED METHOD (TAKES 1.5 MIN TO RUN, RUN ONLY IF YOU HAVE TIME)

#### Loading the fine-tuned Gemma3-1b-it model from HuggingFace

In [ ]:
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig, Gemma3ForCausalLM

##### Loading the pre-trained `gemma-3-1b-it` model

Make sure to get access to the model at https://huggingface.co/google/gemma-3-1b-it before running this cell

In [ ]:
gemma_model_id = "google/gemma-3-1b-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

gemma_model = Gemma3ForCausalLM.from_pretrained(
    "google/gemma-3-1b-it",
    quantization_config=quantization_config,
    device_map="auto"
)

gemma_tokenizer = AutoTokenizer.from_pretrained(gemma_model_id)
gemma_tokenizer.pad_token = gemma_tokenizer.eos_token

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-3-1b-it.
401 Client Error. (Request ID: Root=1-69b77b1e-563792a610580cfc659742d5;e412a0e9-bd41-4796-908a-a52d7cfbd8ab)

Cannot access gated repo for url https://huggingface.co/google/gemma-3-1b-it/resolve/main/config.json.
Access to model google/gemma-3-1b-it is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    gemma_model,
    "mhaiskaa/gemma-finetuned-human-ai-code-classification"
)

tokenizer = AutoTokenizer.from_pretrained(
    "mhaiskaa/gemma-finetuned-human-ai-code-classification"
)

In [ ]:
import json

example1 = json.dumps({
    "result": "Human",
    "reasoning": "No docstrings or type annotations. Any comments that are present may have incorrect punctuation, spelling, or grammar. Code might also be very long"
})

example2 = json.dumps({
    "result": "AI",
    "reasoning": "Frequently uses comments, docstrings, and type annotations. Comments have perfect punctuation, spelling, and grammar"
})

skeleton = json.dumps({"result": "Human/AI", "reasoning": "..."})

#### Add each code snippet from `sample_df` into a prompt structure for inference

The prompt includes examples of the expected JSON output:
1. An example of the output with AI-generated code
2. An example of the output with human-written code
3. A generic skeleton JSON to inform the model that the label is constricted to "Human" or "AI"

After the examples, the code to classify is given.

In [ ]:
def format_example(row):
    prompt = f"""Respond in JSON with keys 'result' (that only contains values 'Human' or 'AI') and 'reasoning' (which explains why you classified the 'result' as 'Human' or 'AI').
    Here are some example models of Code and Completion.

    Human Example:
    Code:
    def _build_url(cls, request, path=None, **changes):
        query_strings = []
        def add_query(key):
            query_strings.append('='.format(key, queries[key])
                                 if queries[key] != '' else key)
        def del_query(key):
            queries.pop(key, None)
        if 'head' in queries:
            add_query('head')
            del_query('head')
        if 'limit' in queries:
            add_query('limit')
            del_query('limit')
        for key in sorted(queries):
            add_query(key)
        scheme = cls._get_forwarded(request, 'proto') or request.url.scheme
        host = cls._get_forwarded(request, 'host') or request.host
        forwarded_path = cls._get_forwarded(request, 'path')
        path = path if path is not None else request.path
        query = '?' + '&'.join(query_strings) if query_strings else ''
        url = ''.format(scheme, host, forwarded_path, path, query)
        return url

    Completion:
    {example1}

    AI Example:
    Code:
    def _build_url(cls, request, path=None, **changes):
      # Parse the original URL
      parsed_url = urlparse(request.url)
        # Get the original queries
        original_queries = dict(qc.split(\"=\") for qc in parsed_url.query.split(\"&\")

        # Update the original queries with the changes
          for key, value in changes.items():
              if value is False:
                # Remove the key if it exists
                original_queries.pop(key, None)
              elif value is not None:
                # Update the key with the new value
                original_queries[key] = value
                # Build the new URL
                new_url = urlunparse((
                  parsed_url.scheme,
                  parsed_url.netloc,
                  path or parsed_url.path,
                  \"\",  # This should be empty for the queries
                  urlencode(original_queries, doseq=True),  # Encode the queries\
                  \"\"  # Fragment is not used
                ))


    Completion:
    {example2}

    The generic template of the Completion is: {skeleton}.

    Think about the patterns that distinguish AI-generated from Human-written code.
    Code to classify:
    {row["code"][:1000]}
"""
    return prompt

In [ ]:
prompts = sample_df.apply(format_example, axis=1).tolist()

#### Testing the model's performance on the above `prompts`

In [ ]:
import torch

model.eval()

results = []

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.0,
        do_sample=False
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    results.append(decoded)

In [ ]:
import json
import re

import json

def extract_model_result(output: str, marker: str = "Code to classify:"):
    """
    Extract the model JSON by looking for the "result" key after a marker.
    Returns a dict with 'result' and 'reasoning' or None.
    """
    try:
        # Only consider text after the last marker
        if marker in output:
            output = output.split(marker)[-1]

        # Find the first occurrence of '"result"' or "'result'"
        idx = output.find('"result"')
        if idx == -1:
            idx = output.find("'result'")
        if idx == -1:
            return None

        # Slice from "result" to the end
        json_str = '{' + output[idx:]

        # Try to parse the JSON from this string
        # Sometimes there is extra trailing text, so remove everything after the last closing brace
        last_brace = json_str.rfind("}")
        if last_brace != -1:
            json_str = json_str[:last_brace+1]

        return json.loads(json_str)
    except Exception as e:
        print("Failed to parse JSON:", e)
        return None

json_results = [extract_model_result(r) for r in results]

In [ ]:
# Normalize the valid labels from the model to account for the model output having different capitalization or extra quotes
label_map = {"Human": "Human", "AI": "AI", "\"Human\"": "Human", "\"AI\"": "AI", "human": "Human", "ai": "AI"}

predictions = []
for r in json_results:
    if r is None:
        predictions.append("Unknown")
    else:
        # Map the result to Human/AI, or Unknown if unexpected
        print("Raw predicted label:", r.get('result'))
        print("Reasoning:", r.get('reasoning'))
        predictions.append(label_map.get(r.get("result"), "Unknown"))

In [ ]:
true_labels = []
for bool_label in sample_df["is_ai"].tolist():
    true_labels.append("AI" if bool_label else "Human")

true_labels

In [ ]:
for i in range(len(true_labels)):
    print("=" * 60)
    print(f"True label : {true_labels[i]}")
    print(f"Prediction : {predictions[i]}")